# This notebook builds a simple FAISS RAG flow.

In [50]:
# Import the main tools we need.
import os
import re
import json
import hashlib
from typing import Dict, List, Tuple
from pathlib import Path

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from huggingface_hub import login

## Import the basic libraries.

In [51]:
# Import more tools and load the env file.
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load values from the root .env file.


True

## Load FAISS, embeddings, and .env values.

In [52]:
# Set the LLM name and start the model.
llm_model_name = "qwen/qwen3-32b"

llm = ChatGroq(
    model=llm_model_name,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
 )

print(f"LLM model: {llm_model_name}")

LLM model: qwen/qwen3-32b


## Find the dataset text files.

In [53]:
# Find the text files in the dataset folder.
datasets_dir = Path(r"C:\projects\learn-rag\Datasets")
txt_files = sorted(datasets_dir.glob("*.txt"))

print(f"Found {len(txt_files)} text file(s) in {datasets_dir}:")
for i, file_path in enumerate(txt_files, start=1):
    print(f"{i}. {file_path.name}")

Found 4 text file(s) in C:\projects\learn-rag\Datasets:
1. amazon_2023.txt
2. amazon_2024.txt
3. microsoft_2023.txt
4. microsoft_2024.txt


## Preview one dataset file.

In [54]:
# Found 4 text file(s) in C:\projects\learn-rag\Datasets:
# 1. amazon_2023.txt
# 2. amazon_2024.txt
# 3. microsoft_2023.txt
# 4. microsoft_2024.txt

# Load the first text file and show a short preview.
if txt_files:
    selected_file = txt_files[0]
    text_data = selected_file.read_text(encoding="utf-8", errors="ignore")
    print(f"\nLoaded file: {selected_file.name}")
    print(f"Character count: {len(text_data):,}")
    print("Preview:\n")
    print(text_data[:500])
else:
    text_data = ""
    print("No .txt files found.")


Loaded file: amazon_2023.txt
Character count: 5,593
Preview:

company_name: Amazon
year: 2023
ticker: AMZN
fiscal_period: FY 2023
headquarters: Seattle, Washington, United States
ceo: Andy Jassy
founded: 1994
industry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI
revenue_usd_billions: 574.7
net_income_usd_billions: 30.4
employee_count: 1525000

Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was found


## Check all dataset files.

In [ ]:
# Read each dataset file once so we can inspect the corpus coverage.
for file_path in txt_files:
    file_text = file_path.read_text(encoding="utf-8", errors="ignore")
    parts = file_path.stem.split("_")
    company = parts[0] if len(parts) > 0 else "unknown"
    year = parts[1] if len(parts) > 1 else "unknown"
    print(len(file_text))

5593
5329
5253
5604


## Split each file into chunks with metadata.

In [70]:
# Split the text into token-aware chunks and keep source metadata.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=250,
    chunk_overlap=50,
)

# This single-file example is useful for learning, but the final docs list
# below is rebuilt from all files so the vector store covers the full dataset.
docs = splitter.create_documents(
    texts=[text_data],
    metadatas=[{
        "source": str(selected_file),
        "doc_name": selected_file.name,
        "year": "2023",
        "company": "amazon",
    }],
)

docs = []

# Build one chunked document list across every dataset file.
for file_path in txt_files:
    file_text = file_path.read_text(encoding="utf-8", errors="ignore")
    parts = file_path.stem.split("_")
    company = parts[0] if len(parts) > 0 else "unknown"
    year = parts[1] if len(parts) > 1 else "unknown"
    docs.extend(splitter.create_documents(
        texts=[file_text],
        metadatas=[{
            "source": str(file_path),
            "doc_name": file_path.name,
            "year": year,
            "company": company,
        }],
    ))

In [57]:
docs

[Document(metadata={'source': 'C:\\projects\\learn-rag\\Datasets\\amazon_2023.txt', 'doc_name': 'amazon_2023.txt', 'year': '2023', 'company': 'amazon'}, page_content="company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000\n\nAmazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.\n\nIn 2023, Amazon reported approximately $574.7 billion in revenu

## Set the embedding model name and token.

In [58]:
# Set the embedding model name and load the token.
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

## Load the embedding model.

In [59]:
# Load the sentence embedding model.
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(embedding_model_name, device="cpu")

print(f"Loaded embedder: {embedding_model_name}")
print(f"Embedding dimension: {embedder.get_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1672.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedder: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


## Turn the chunks into vectors.

In [71]:
# Make an adapter so LangChain FAISS can call the sentence-transformer model.
from langchain_core.embeddings import Embeddings

class SentenceTransformerAdapter(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        # Normalize embeddings so vector search uses comparable lengths.
        return self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0].tolist()

embedding_adapter = SentenceTransformerAdapter(embedder)

chunk_texts = [doc.page_content for doc in docs]
vectors = embedding_adapter.embed_documents(chunk_texts)
print(f"Chunk vectors shape: {len(vectors)} x {len(vectors[0])}")

Chunk vectors shape: 20 x 384


## Preview the vectors.

In [61]:
# Show the vectors.
vectors

[[0.004097082186490297,
  -0.07407250255346298,
  -0.022514423355460167,
  -0.013816793449223042,
  0.043621975928545,
  -0.03130115941166878,
  -0.003513065865263343,
  0.01308564841747284,
  0.044288914650678635,
  0.0371445007622242,
  -0.029770921915769577,
  0.0606662780046463,
  0.060885537415742874,
  -0.02430819906294346,
  0.007720400579273701,
  -0.03914972394704819,
  0.004225677344948053,
  -0.1342686414718628,
  -0.07496418803930283,
  -0.12339319288730621,
  0.017941731959581375,
  0.02836916223168373,
  -0.011320285499095917,
  -0.04845316708087921,
  -0.028825119137763977,
  0.03607704117894173,
  -0.046710338443517685,
  -0.015868667513132095,
  -0.03766854107379913,
  -0.046881914138793945,
  0.016512013971805573,
  -0.02569892443716526,
  0.10196053236722946,
  0.05558622255921364,
  -0.04329854995012283,
  0.012452446855604649,
  -0.002036143559962511,
  -0.09347975999116898,
  0.0030975425615906715,
  0.005919670220464468,
  0.013442703522741795,
  0.00278603076003

## Create the FAISS store.

In [62]:
# Create the FAISS vector store.
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = embedder.get_embedding_dimension()
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embedding_adapter,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

print(f"Vector store initialized with dimension: {embedding_dim}")

Vector store initialized with dimension: 384


## The FAISS store is ready.

## Add chunk embeddings to FAISS.

In [72]:
# Add the chunk texts and their vectors into the FAISS store.
from uuid_utils import uuid4

chunk_metadatas = [doc.metadata for doc in docs]
# Each chunk gets its own id so the vector, text, and metadata stay linked.
uuids = [str(uuid4()) for _ in range(len(vectors))]
text_embeddings = list(zip(chunk_texts, vectors))

vector_store.add_embeddings(
    text_embeddings=text_embeddings,
    metadatas=chunk_metadatas,
    ids=uuids,
)

print(f"Added {len(uuids)} embeddings to FAISS")

Added 20 embeddings to FAISS


## Save and reload the FAISS store.

In [64]:
# Save the FAISS store and load it again.
store_dir = Path.cwd() / "faiss_store"
store_dir.mkdir(parents=True, exist_ok=True)

vector_store.save_local(folder_path=str(store_dir))
print(f"Saved vector store to: {store_dir}")

loaded_vector_store = FAISS.load_local(
    folder_path=str(store_dir),
    embeddings=embedding_adapter,
    allow_dangerous_deserialization=True,
)

Saved vector store to: c:\projects\learn-rag\vectorDB\faiss_store


## Compare search with and without metadata filters.

In [73]:
# Compare plain vector search with metadata-filtered search.
all_docs = vector_store.similarity_search(
    "microsoft 2023",
    k=len(vectors),
    fetch_k=len(vectors),
)

# This filter keeps only chunks that match the target company and year.
filtered_docs = vector_store.similarity_search(
    "microsoft 2023",
    k=len(vectors),
    fetch_k=len(vectors),
    filter={"company": "microsoft", "year": "2023"},
)

print(f"Docs before metadata filter: {len(all_docs)}")
print(f"Docs after metadata filter: {len(filtered_docs)}")

Docs before metadata filter: 20
Docs after metadata filter: 10


In [74]:
# Search the store with a metadata filter and print the best matching chunks.
query = "what is revenue of microsoft in 2023 ?"

results = vector_store.similarity_search(
    query,
    k=5,
    filter={"company": "microsoft", "year": "2023"},
)

print(f"Top {len(results)} retrieved chunks from the loaded store:")
answers = []
for i, doc in enumerate(results, start=1):
    answers.append(doc.page_content)
    print(f"\nResult {i}:\n{doc.page_content[:1000]}")

Top 5 retrieved chunks from the loaded store:

Result 1:
company_name: Microsoft
year: 2023
ticker: MSFT
fiscal_period: FY 2023
headquarters: Redmond, Washington, United States
ceo: Satya Nadella
founded: 1975
industry: Software, cloud computing, AI, productivity software, gaming, devices
revenue_usd_billions: 211.9
net_income_usd_billions: 72.4
employee_count: 221000

Microsoft is a global technology company focused on software, cloud infrastructure, developer tools, productivity platforms, AI, gaming, and enterprise services. Founded in 1975, the company operates across major business groups that include Productivity and Business Processes, Intelligent Cloud, and More Personal Computing. Satya Nadella served as chairman and chief executive officer in fiscal year 2023.

In fiscal year 2023, Microsoft reported a record $211.9 billion in revenue and $72.4 billion in net income. The company highlighted strong demand for Azure, Office 365 Commercial, LinkedIn, and Dynamics, while some con

## Load the reranker model files.

In [75]:
# Load the Hugging Face reranker model and tokenizer.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
# Reuse the optional token for gated or rate-limited model access.
token_arg = hf_token if hf_token else None

model = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    token=token_arg,
)
tokenizer = AutoTokenizer.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    token=token_arg,
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2071.41it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Create the reranker instance.

In [76]:
# CrossEncoder wraps the model so we can score question-answer pairs directly.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", token=token_arg)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4769.56it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Rerank the retrieved chunks.

In [77]:
# Rerank the retrieved chunks and keep the highest-scoring answers.
import torch

def top_k_rerank(question, answers, top_k=5):
    """Pick the best answers with the reranker."""
    if not answers:
        return []

    # Keep top_k inside the list size.
    top_k = max(1, min(top_k, len(answers)))

    # Score each question-answer pair with the cross-encoder.
    features = tokenizer(
        [question] * len(answers),
        answers,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    model.eval()
    with torch.no_grad():
        logits = model(**features).logits.squeeze(-1)

    # Convert raw logits into easy-to-read relevance scores.
    scores = torch.sigmoid(logits).tolist()

    ranked = sorted(
        [{"answer": ans, "score": float(score)} for ans, score in zip(answers, scores)],
        key=lambda x: x["score"],
        reverse=True,
    )
    return ranked[:top_k]

# Test the reranker with the current answers.
question = "what is revenue of microsoft in 2023 ?"
top_results = top_k_rerank(question, answers, top_k=10)
top_results

[{'answer': 'company_name: Microsoft\nyear: 2023\nticker: MSFT\nfiscal_period: FY 2023\nheadquarters: Redmond, Washington, United States\nceo: Satya Nadella\nfounded: 1975\nindustry: Software, cloud computing, AI, productivity software, gaming, devices\nrevenue_usd_billions: 211.9\nnet_income_usd_billions: 72.4\nemployee_count: 221000\n\nMicrosoft is a global technology company focused on software, cloud infrastructure, developer tools, productivity platforms, AI, gaming, and enterprise services. Founded in 1975, the company operates across major business groups that include Productivity and Business Processes, Intelligent Cloud, and More Personal Computing. Satya Nadella served as chairman and chief executive officer in fiscal year 2023.\n\nIn fiscal year 2023, Microsoft reported a record $211.9 billion in revenue and $72.4 billion in net income. The company highlighted strong demand for Azure, Office 365 Commercial, LinkedIn, and Dynamics, while some consumer and device categories we